# Control the physical bioreactor using RL

In [1]:
import socket
import time
import csv
from datetime import datetime
from scipy.io import savemat,loadmat

In [2]:
# Arduino AP IP address and port
arduino_ip = "192.168.1.1"  # Use the static IP address you set for the Arduino
arduino_port = 80
LINE_CLEAR = '\x1b[2K'

# Create a TCP/IP socket
client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# # Connect to the Arduino AP
try:
    client_socket.connect((arduino_ip, arduino_port))
    print("Connection Successful")
except:
    print("Connection Unsuccessful")

Connection Successful


In [ ]:
# turb_measure = "TurMeas_1000_158"
# turbidity = "Sample2_800_9000"

# client_socket.sendall(turbidity.encode())
# client_socket.close()

In [ ]:
from datetime import datetime, timedelta

def print_hello_world_real_time(interval_minutes):
    """
    Continuously checks the time and prints "Hello, World!" after every n minutes in real time.
    
    Args:
        interval_minutes (int): The interval in minutes between messages.
    """
    next_time = datetime.now() + timedelta(minutes=interval_minutes)
    add_substrate = "Sample2_800_9000"
    turbidity = "TurMeas_1000_153"
    # pump = "Pump1_1_0"
    while True:
        current_time = datetime.now()
        if current_time >= next_time:
            # Measure Turbidity
            print("-------------------------------------------------------------------------------------------------")
            print(f"{current_time.strftime('%Y-%m-%d %H:%M:%S')} - Sampling and Measuring Turbidity {turbidity}")
            client_socket.sendall(turbidity.encode())
            # Received_data=client_socket.recv(3000)
            # dataN = int(client_socket.recv(1000))
            # print("Turbidity DataN :"+str(dataN)+"\n")
            Tur_Time = []
            Tur_Sig = []
            Tur_data = client_socket.recv(1000)
            data_str = Tur_data.decode("utf-8")

            # data_str =  Received_data.decode("utf-8")
            #print(f"This is data from reciever: ",data_str)
            print("-------------------------------------------------------------------------------------------------")

            print("-------------------------------------------------------------------------------------------------")
            print(f"{current_time.strftime('%Y-%m-%d %H:%M:%S')} - Adding Substrate {add_substrate}")
            client_socket.sendall(add_substrate.encode())
            print("-------------------------------------------------------------------------------------------------")

            #client_socket.sendall(command.encode())
            next_time += timedelta(minutes=interval_minutes)
        

# Set the interval in minutes
n = 0.1  # Replace with your desired interval  6 seconds
print_hello_world_real_time(n)


In [3]:
from datetime import datetime, timedelta

def execute_commands(interval_seconds, offset_seconds):
    """
    Executes turbidity command every `interval_seconds` and add substrate command 
    `offset_seconds` after each turbidity command.

    Args:
        interval_seconds (int): Time interval (in seconds) between consecutive turbidity commands.
        offset_seconds (int): Time offset (in seconds) to run the add substrate command after turbidity.
    """
    add_substrate = "Sample2_800_9000"
    turbidity = "TurMeas_1000_153"
    
    # Initialize timing
    next_turbidity_time = datetime.now()
    next_add_substrate_time = next_turbidity_time + timedelta(seconds=offset_seconds)

    while True:
        current_time = datetime.now()

        # Execute turbidity command
        if current_time >= next_turbidity_time:
            print("-------------------------------------------------------------------------------------------------")
            print(f"{current_time.strftime('%Y-%m-%d %H:%M:%S')} - Sampling and Measuring Turbidity {turbidity}")
            client_socket.sendall(turbidity.encode())
            Tur_data = client_socket.recv(1000)
            data_str = Tur_data.decode("utf-8")
            print(f"Turbidity Data: {data_str}")
            print("-------------------------------------------------------------------------------------------------")
            
            # Schedule next turbidity and add substrate timings
            next_turbidity_time += timedelta(seconds=interval_seconds)
            next_add_substrate_time = current_time + timedelta(seconds=offset_seconds)

        # Execute add substrate command
        if current_time >= next_add_substrate_time:
            print("-------------------------------------------------------------------------------------------------")
            print(f"{current_time.strftime('%Y-%m-%d %H:%M:%S')} - Adding Substrate {add_substrate}")
            client_socket.sendall(add_substrate.encode())
            print("-------------------------------------------------------------------------------------------------")
            
            # Ensure the add substrate command doesn't keep firing until the next turbidity cycle
            next_add_substrate_time += timedelta(seconds=interval_seconds)

# Set the intervals (in seconds)
interval_seconds = 30  # Turbidity every 30 seconds
offset_seconds = 10    # Add substrate 10 seconds after turbidity
execute_commands(interval_seconds, offset_seconds)


-------------------------------------------------------------------------------------------------
2024-11-22 10:47:53 - Sampling and Measuring Turbidity TurMeas_1000_153
Turbidity Data: 201

-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
2024-11-22 10:48:03 - Adding Substrate Sample2_800_9000
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
2024-11-22 10:48:23 - Sampling and Measuring Turbidity TurMeas_1000_153
Turbidity Data: 1001_4095
1010_4095
1020_4095
1030_4095
1040_4095
1050_4095
1060_4095
1070_4095
1080_4095
1090_4095
1100_4095
1110_4095
1120_4095
1130_4095
1140_4095
1150_4095
1160_4095
1170_4095
1180_4095
1190_4095
1200_4095
1210_4095
1220_4095
1230_4095
1240_4095
1250_4095
1260_4095
127

KeyboardInterrupt: 